# Bearing Diagnostics Step-by-Step

This notebook demonstrates the bearing fault detection pipeline:
1. Look up bearing specifications from the catalog
2. Compute characteristic fault frequencies (BPFO, BPFI, BSF, FTF)
3. Run envelope spectrum analysis
4. Check for fault peaks with confidence scoring
5. Get a full diagnostic summary

In [ ]:
import numpy as np
from pathlib import Path
import sys, os

project_root = Path('.').resolve().parent
sys.path.insert(0, str(project_root / 'src'))
os.chdir(project_root)

## Step 1: Bearing Catalog Lookup

The catalog contains 22+ bearings from SKF, FAG, Timken, and NSK.

In [ ]:
from predictive_maintenance_mcp.diagnostics import (
    lookup_bearing, compute_fault_frequencies, list_catalog_bearings
)

# List available bearings
catalog = list_catalog_bearings()
print(f'Catalog contains {len(catalog)} bearings:')
for b in catalog[:5]:
    print(f'  {b}')
print(f'  ... and {len(catalog)-5} more')

In [ ]:
# Look up a specific bearing
bearing = lookup_bearing('6205')
if bearing:
    print(f'Bearing: {bearing.get("designation", "6205")}')
    for k, v in bearing.items():
        if k != 'designation':
            print(f'  {k}: {v}')

## Step 2: Compute Fault Frequencies

In [ ]:
rpm = 1800  # shaft speed

freqs = compute_fault_frequencies('6205', rpm)
if freqs:
    print(f'Fault frequencies at {rpm} RPM:')
    for name, val in freqs.items():
        if isinstance(val, (int, float)):
            print(f'  {name}: {val:.2f} Hz')

## Step 3: Generate a Faulty Signal

In [ ]:
fs = 10000
duration = 2.0
t = np.arange(0, duration, 1/fs)

# Use actual BPFO from catalog
bpfo = freqs['BPFO'] if freqs else 120.0
f_shaft = rpm / 60.0

# Outer race fault signal
signal = (
    0.3 * np.sin(2 * np.pi * f_shaft * t)
    + 0.4 * np.sin(2 * np.pi * bpfo * t)
    + 0.2 * np.sin(2 * np.pi * 2 * bpfo * t)
    + 0.1 * np.sin(2 * np.pi * 3 * bpfo * t)
    + 0.15 * np.random.randn(len(t))
)

print(f'Synthetic outer race fault signal: {len(signal)} samples')
print(f'BPFO = {bpfo:.2f} Hz, shaft = {f_shaft:.1f} Hz')

## Step 4: Envelope Spectrum Analysis

In [ ]:
from predictive_maintenance_mcp.signal_processing import compute_envelope_spectrum

env = compute_envelope_spectrum(signal, fs, frequency_range=(50, 4000), num_peaks=15)

print('Envelope spectrum peaks:')
for i, peak in enumerate(env['top_peaks'][:8], 1):
    # Check if near a fault frequency
    note = ''
    if freqs:
        for name, fval in freqs.items():
            if isinstance(fval, (int, float)) and abs(peak['frequency_hz'] - fval) / fval < 0.05:
                note = f'  <-- {name}'
                break
        for n in range(2, 4):
            if isinstance(freqs.get('BPFO'), (int, float)):
                if abs(peak['frequency_hz'] - n * freqs['BPFO']) / (n * freqs['BPFO']) < 0.05:
                    note = f'  <-- {n}x BPFO'
    print(f'  {i}. {peak["frequency_hz"]:8.2f} Hz  ({peak["magnitude_db"]:+.1f} dB){note}')

## Step 5: Fault Detection with Confidence Scoring

In [ ]:
from predictive_maintenance_mcp.diagnostics import check_all_bearing_faults

faults = check_all_bearing_faults(signal, fs, bearing_id='6205', rpm=rpm)

print(f'Bearing fault check results ({len(faults)} fault types):')
for fault in faults:
    status = 'DETECTED' if fault.get('detected') else 'not detected'
    conf = fault.get('confidence', 'N/A')
    print(f'  {fault["fault_type"]:15s}: {status} (confidence: {conf})')
    if fault.get('detected'):
        print(f'    Expected freq: {fault.get("expected_frequency", "?")} Hz')
        print(f'    Peak amplitude: {fault.get("peak_amplitude", "?")}')

## Summary

This notebook demonstrated:
1. Looking up bearing specs from the 22+ bearing catalog
2. Computing characteristic fault frequencies (BPFO, BPFI, BSF, FTF)
3. Envelope spectrum analysis to reveal fault-related frequencies
4. Automated fault detection with confidence scoring

**Next**: See `03_condition_monitoring.ipynb` for the full diagnosis pipeline.